In [1]:
import pandas as pd
import json
import rdkit
import os
from tqdm import tqdm

In [48]:
df1 = pd.read_csv('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datasets/2p2idb/2p2idb_2024-04-08.csv', sep=";")
df2 = pd.read_csv('ppi_inhibitor_mapping.csv')

df = df2.merge(df1[['PDBProtProt', 'PDBProtLig', 'InChI']], on=['PDBProtProt', 'PDBProtLig'], how='left')
df = df[df['PDBProtProt'] != 'na']

## Process PPI inhibitors

In [3]:
ccd_code_to_smiles = pd.read_csv('/n/holylabs/LABS/mzitnik_lab/Users/afang/ATOMICA/src/atomica/data/converter/pdb_chemical_components_smiles.txt', sep="\t", names=['smiles', 'ccd', 'name'])
ccd_code_to_smiles = ccd_code_to_smiles.set_index('ccd')['smiles'].to_dict()

In [4]:
smiles_list = []
for smiles, ligand in zip(df['SMILES'], df['Ligand']):
    if not pd.isna(smiles):
        smiles_list.append(smiles)
        continue
    # try inchi to smiles
    try:
        smiles = rdkit.Chem.MolToSmiles(rdkit.Chem.MolFromInchi(ligand))
    except:
        ligand = ligand.lstrip("'")
        if ligand in ccd_code_to_smiles:
            smiles = ccd_code_to_smiles[ligand]
        else:
            smiles = None
    smiles_list.append(smiles)

df['SMILES'] = smiles_list
df = df[df['PDBProtProt']!='na']

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 

[10:57:35] ERROR: 



In [5]:
print("Number of entries: ", len(df))
df.head()

Number of entries:  2096


,2P2IDB_ID,Family,Target,Partner,PDBProtProt,PDBProtLig,Chain_Target,Chain_Partner,ChainID_Ligand,Ligand,SMILES,MolecularWeight,HeavyAtoms,UniProt_Target,UniProt_Partner,InChI
0,10101-0001,BCL2/BAX,BCL2,BAX,2XA0,1YSW,A,C,A,43B,c1ccc(cc1)CCc2nc3cc(ccc3s2)c4ccc(cc4)C(=O)NS(=...,694.842,48,P10415,Q07813,1YSW_43B_A_1000
1,10101-0002,BCL2/BAX,BCL2,BAX,2XA0,2O21,A,C,A,43B,c1ccc(cc1)CCc2nc3cc(ccc3s2)c4ccc(cc4)C(=O)NS(=...,694.842,48,P10415,Q07813,2O21_43B_A_1000
2,10101-0003,BCL2/BAX,BCL2,BAX,2XA0,2O22,A,C,A,LIU,CC1(CCN(CC1)c2ccc(cc2)C(=O)NS(=O)(=O)c3ccc(c(c...,596.761,41,P10415,Q07813,2O22_LIU_A_1000
3,10101-0004,BCL2/BAX,BCL2,BAX,2XA0,2O2F,A,C,A,LI0,CC(C)(CSc1ccccc1)Nc2ccc(cc2[N+](=O)[O-])S(=O)(...,688.856,48,P10415,Q07813,2O2F_LI0_A_1000
4,10101-0005,BCL2/BAX,BCL2,BAX,2XA0,2W3L,A,C,A,DRO,Cc1c(c(nn1c2ccccc2C(=O)N3Cc4ccccc4C[C@H]3CN)C(...,576.087,41,P10415,Q07813,2W3L_DRO_A_1166


In [16]:
from __future__ import annotations

from typing import List
import numpy as np
from pathlib import Path

import biotite.structure as struc
import biotite.structure.io.pdb as pdb
import biotite.structure.io.pdbx as pdbx


def get_lig_resi(pdb_path: str, chain: str, lig_code: str) -> List[int]:
    """
    Find residue IDs (resi) for a ligand in a given chain.
    Supports both PDB and mmCIF files.

    Parameters
    ----------
    pdb_path : str
        Path to PDB or CIF file.
    chain : str
        Chain ID (e.g. "A")
    lig_code : str
        Ligand residue name (e.g. "ATP")

    Returns
    -------
    List[int]
        Sorted list of residue IDs matching ligand and chain.
    """

    lig_code = lig_code.strip().upper()
    chain = chain.strip()

    pdb_path = Path(pdb_path)
    suffix = pdb_path.suffix.lower()

    # -------- Load structure --------
    if suffix in [".pdb", ".ent"]:
        file = pdb.PDBFile.read(pdb_path)
        arr = file.get_structure(model=1)

    elif suffix in [".cif", ".mmcif"]:
        file = pdbx.PDBxFile.read(pdb_path)
        arr = pdbx.get_structure(file, model=1)

    else:
        raise ValueError(f"Unsupported file type: {suffix}")

    # -------- Select ligand --------
    mask = (arr.chain_id == chain) & (arr.res_name == lig_code)

    if not np.any(mask):
        return []

    res_ids = np.unique(arr.res_id[mask])

    return sorted(res_ids.tolist())


def check_chain_exists(pdb_path: str, chain: str) -> bool:
    pdb_path = Path(pdb_path)
    suffix = pdb_path.suffix.lower()
    if suffix in [".pdb", ".ent"]:
        file = pdb.PDBFile.read(pdb_path)
        arr = file.get_structure(model=1)
    elif suffix in [".cif", ".mmcif"]:
        file = pdbx.PDBxFile.read(pdb_path)
        arr = pdbx.get_structure(file, model=1)
    unique_chains = np.unique(arr.chain_id)
    return chain in unique_chains, unique_chains

In [18]:
ppi_inhibitors = df[['2P2IDB_ID', 'PDBProtLig', 'Family', 'Chain_Target', 'ChainID_Ligand', 'Ligand', 'SMILES']].drop_duplicates()
ppi_inhibitors['SMILES']
print("Number of PPI inhibitors: ", len(ppi_inhibitors))
print("Number of families: ", len(ppi_inhibitors['Family'].unique()))

ppi_inhibitors.rename(columns={
    '2P2IDB_ID': 'pdb_id',
    'PDBProtLig': 'pdb_code',
    'Chain_Target': 'chain1',
    'ChainID_Ligand': 'chain2',
    'Ligand': 'lig_code',
    'SMILES': 'lig_smiles',
}, inplace=True)

# Match to CIF/PDB files
ppi_inhibitors['pdb_path'] = ppi_inhibitors['pdb_code'].apply(lambda x: f'/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datasets/2p2idb/2p2idb_2024-04-08_cifs/{x}.cif')
mask = ~ppi_inhibitors['pdb_path'].apply(os.path.exists)
ppi_inhibitors.loc[mask, 'pdb_path'] = (
    ppi_inhibitors.loc[mask, 'pdb_code']
    .apply(lambda x: f'/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datasets/2p2idb/2p2idb_2024-04-08_pdbs/{x}.pdb')
)
mask = ~ppi_inhibitors['pdb_path'].apply(os.path.exists)
print("Missing files for: ", mask.sum(), "out of", len(ppi_inhibitors))

ppi_inhibitors = ppi_inhibitors[~mask]

ppi_inhibitors_chains = ppi_inhibitors.apply(
    lambda row: pd.Series(
        check_chain_exists(
            row['pdb_path'],
            row['chain1']
        ),
        index = ['chain1_exists', 'unique_chains']
    ),
    axis=1
)
ppi_inhibitors[['chain1_exists', 'unique_chains']] = ppi_inhibitors_chains

ppi_inhibitors['lig_resi'] = ppi_inhibitors.apply(lambda row: get_lig_resi(row['pdb_path'], row['chain2'], row['lig_code']), axis=1)
ppi_inhibitors['lig_resi'].apply(len).value_counts()

Number of PPI inhibitors:  1976
Number of families:  39
Missing files for:  7 out of 1976


lig_resi
1    1861
2      86
0      17
3       3
4       1
5       1
Name: count, dtype: int64

In [ ]:
# try infill chain1
mask = ppi_inhibitors['chain1_exists'] == False
print("Number of PPIs with missing chain1: ", mask.sum())

def infill_chain1(row):
    # Could be a cif file and the chain is AAA instead of A
    chain3 = ''.join([row['chain1'] for _ in range(3)])
    if len(row['chain1']) == 1 and (chain3 in row['unique_chains']):
        return chain3
    # Use the ligand chain if it exists
    if row['chain2'] in row['unique_chains']:
        return row['chain2']
    # If there is only one chain, use it
    if len(row['unique_chains']) == 1:
        return row['unique_chains'][0]
    return None 

ppi_inhibitors_with_missing_chain1 = ppi_inhibitors[mask].copy()
ppi_inhibitors_with_missing_chain1['chain1'] = ppi_inhibitors_with_missing_chain1.apply(infill_chain1, axis=1)

Number of PPIs with missing chain1:  553


In [23]:
ppi_inhibitors.loc[mask, 'chain1'] = ppi_inhibitors_with_missing_chain1['chain1']
# how many are still missing chain1
print("Number of PPIs with missing chain1: ", len(ppi_inhibitors[ppi_inhibitors['chain1'].isna()]))
ppi_inhibitors = ppi_inhibitors[ppi_inhibitors['chain1'].notna()]

Number of PPIs with missing chain1:  0


In [24]:
ppi_inhibitors[ppi_inhibitors['lig_resi'].apply(len) > 2]

,pdb_id,pdb_code,Family,chain1,chain2,lig_code,lig_smiles,pdb_path,chain1_exists,unique_chains,lig_resi
558,20401-0006,3NF8,INTEGRASE/LEDGF,A,A,CDQ,c1cc2c(cc1Cl)CC(=O)N2Cc3ccc4c(c3C(=O)O)OCCO4,/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datase...,True,"[A, B]","[225, 247, 267, 277]"
935,30401-0027,5QXN,ATAD2/H4,A,A,RHG,C1CNC[C@@]23CN(C[C@@H]2C[C@@H]1O3)S(=O)(=O)C(F...,/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datase...,True,[A],"[1201, 1202, 1203]"
960,30401-0052,6HI5,ATAD2/H4,A,A,G6E,C[C@H](C(=O)Nc1nc(c(s1)c2cc(cnc2)N)C(=O)C)N,/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datase...,True,[A],"[1203, 1204, 1205]"
4692,30205-0128,5E0R,BRD4-1/H4,A,A,5J5,c1cc(ccc1/N=N/c2ccc(c(c2)S(=O)(=O)O)NC(=O)CCl)...,/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datase...,True,[A],"[201, 202, 203, 204, 205]"
4913,30205-0347,7K6G,BRD4-1/H4,A,A,VYJ,CCOc1cc(ccc1Nc2ncc3c(n2)N(c4ccccc4C(=O)N3C)C5C...,/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datase...,True,"[A, B]","[201, 202, 203]"


In [25]:
ppi_inhibitors_to_process = ppi_inhibitors[ppi_inhibitors['lig_resi'].apply(len) > 0].explode('lig_resi')
ppi_inhibitors_to_process.to_csv('PPI_inhibitors_v3.csv', index=False)
print("Number of PPI inhibitors to process: ", len(ppi_inhibitors_to_process))

Number of PPI inhibitors to process:  2051


## Process PPIs

Sample 1000 partner points on the surface of the PPI inhibitor with `surface_sampler/process_2p2idb.py`.

Total of 39 PPI structures/families. Some have protein-inhibitor complexes but the PPI structure is "na", these are discarded.

In [60]:
ppis = df[['PDBProtProt', 'Family', 'Chain_Target', 'Chain_Partner']].drop_duplicates().reset_index(drop=True)
print("Number of PPIs: ", len(ppis))
print("Number of families: ", len(ppis['Family'].unique()))


inhibitor_families = set(ppi_inhibitors_to_process['Family'].unique())
print("Number of overlapping inhibitor families: ", len(inhibitor_families & set(ppis['Family'].unique())))

Number of PPIs:  39
Number of families:  39
Number of overlapping inhibitor families:  39


In [50]:
from atomica.data.converter.pdb_to_list_blocks import pdb_to_list_blocks
from atomica.data.dataset import blocks_to_data

all_blocks = []
all_pdb_indexes = []
for i, row in ppis.iterrows():
    pdb_id = row['PDBProtProt']
    chain = row['Chain_Partner']
    pdb_path = f'/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datasets/2p2idb/2p2idb_2024-04-08_cifs/{pdb_id}.cif'
    if not os.path.exists(pdb_path):
        pdb_path = f'/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datasets/2p2idb/2p2idb_2024-04-08_pdbs/{pdb_id}.pdb'
    if not os.path.exists(pdb_path):
        print(f"File {pdb_path} does not exist")
        all_blocks.append([])
        all_pdb_indexes.append([])
        continue

    if not check_chain_exists(pdb_path, chain):
        print(f"Chain {chain} does not exist in {pdb_path}")
        all_blocks.append([])
        all_pdb_indexes.append([])
        continue
    
    blocks, pdb_indexes = pdb_to_list_blocks(pdb_path, [chain], return_indexes=True)
    blocks = sum(blocks, [])
    pdb_indexes = sum(pdb_indexes, [])
    all_blocks.append(blocks)
    all_pdb_indexes.append(pdb_indexes)

ppis['blocks'] = all_blocks
ppis['pdb_indexes'] = all_pdb_indexes
ppis['num_blocks'] = ppis['blocks'].apply(len)

In [57]:
num_blocks_cutoffs = [0, 10, 30, 1000]
ppis['num_blocks_cutoff'] = pd.cut(ppis['num_blocks'], bins=num_blocks_cutoffs, labels=num_blocks_cutoffs[1:])
ppis.groupby('num_blocks_cutoff', observed=False).agg({'num_blocks': ['min', 'max', 'count'], 'Family': list})

num_blocks             \
                         min  max count   
num_blocks_cutoff                         
10                         2   10    16   
30                        11   26    15   
1000                      82  462     8   

                                                              Family  
                                                                list  
num_blocks_cutoff                                                     
10                 [CIAP1-BIR3/CASPASE-9, WDR5-2/MYC, XIAP-BIR2/C...  
30                 [BCL2/BAX, BCLXL/BAK, DCN1/UBC12, HDM2/P53, KE...  
1000               [XIAP-BIR3/SMAC, HPV-E2/E1, HRAS/SOS1, IL-2/IL...

Looking at the number of residues in the PPIs, they vary a lot by length:
* 1-9 residues: discard
* 10-30 residues: keep but run retreival at a per residue level
* \>30 residues: run retrieval at the sample point level

### Large partner proteins
For large partner proteins ( \>30 residues), we sample points on the surface of the inhibitor and retrieve blocks around them.

In [96]:
min_num_blocks = 30
interface_radius = 16
min_num_blocks_near_point = 8

display(ppis[ppis['num_blocks'] > 30])

items = []
num_blocks = []
for _, row in ppis[ppis['num_blocks'] > min_num_blocks].iterrows():
    points_file = f"/n/holylfs06/LABS/mzitnik_lab/Lab/afang/datasets/2p2idb/surface_mesh/{row['PDBProtProt']}_chain{row['Chain_Partner']}_points.xyz"
    if not os.path.exists(points_file):
        print(f"File {points_file} does not exist")
        continue
    
    block_coords = np.array([b.coords for b in row['blocks']])
    points = pd.read_csv(points_file, sep=" ", header=None, skiprows=2)
    for point_idx, point in points.iterrows():
        _, x, y, z = point
        blocks_dist_to_point = np.linalg.norm(block_coords - np.array([x, y, z]), axis=1)
        mask = blocks_dist_to_point < interface_radius
        if mask.sum() < min_num_blocks_near_point:
            print(f"Point {point_idx} only matches {mask.sum()} blocks")
            continue
        
        blocks_near_point = [b for b, m in zip(row['blocks'], mask) if m]
        pdb_indexes_near_point = [p for p, m in zip(row['pdb_indexes'], mask) if m]
        data = blocks_to_data(blocks_near_point)
        pdb_indexes_map = {}
        pdb_indexes_map.update(dict(zip(range(1,len(blocks_near_point)+1), pdb_indexes_near_point))) # map block index to pdb index, +1 for global block)
        items.append({
            **data,
            'block_to_pdb_indexes': json.dumps(pdb_indexes_map),
            'id': f"{row['PDBProtProt']}_{row['Chain_Partner']}_{point_idx}",
        })
        num_blocks.append(len(blocks_near_point))

print("Average number of blocks: ", np.mean(num_blocks), "min: ", np.min(num_blocks), "max: ", np.max(num_blocks))
print("Number of items: ", len(items))
pd.DataFrame(items).to_parquet('PPI_Partner_interface_points.parquet')

,PDBProtProt,Family,Chain_Target,Chain_Partner,blocks,pdb_indexes,num_blocks
500,1G73,XIAP-BIR3/SMAC,C,A,"[(N, CA, C, O, CB), (N, CA, C, O, CB, CG1, CG2...","[A_1, A_2, A_3, A_4, A_5, A_6, A_7, A_8, A_9, ...",157
534,1TUE,HPV-E2/E1,B,A,"[(N, CA, C, O, CB, OG), (N, CA, C, O), (N, CA,...","[A_425, A_426, A_427, A_428, A_429, A_430, A_4...",201
535,1BKD,HRAS/SOS1,R,S,"[(N, CA, C, O, CB, CG, CD, NE, CZ, NH1, NH2), ...","[S_568, S_569, S_570, S_571, S_572, S_573, S_5...",439
543,1Z92,IL-2/IL-2R,A,B,"[(N, CA, C, O, CB, CG, OD1, OD2), (N, CA, C, O...","[B_-1, B_0, B_1, B_2, B_3, B_4, B_5, B_6, B_7,...",123
553,2B4J,INTEGRASE/LEDGF,A,C,"[(N, CA, C, O), (N, CA, C, O, CB, OG), (N, CA,...","[C_345, C_346, C_347, C_348, C_349, C_350, C_3...",82
680,6EPL,KRAS/SOS1,R,S,"[(N, CA, C, O, CB, CG, CD, OE1, OE2), (N, CA, ...","[S_565, S_566, S_567, S_568, S_569, S_570, S_5...",462
821,1TNF,TNFA/TNFA,A,B,"[(N, CA, C, O, CB), (N, CA, C, O, CB, OG1, CG2...","[B_6, B_7, B_8, B_9, B_10, B_11, B_12, B_13, B...",152
835,1TNR,TNFR1A/TNFB,R,A,"[(N, CA, C, O, CB, CG, CD, CE, NZ), (N, CA, C,...","[A_28, A_29, A_30, A_31, A_32, A_33, A_34, A_3...",144


Point 0 only matches 5 blocks
Point 185 only matches 7 blocks
Point 261 only matches 7 blocks
Point 282 only matches 6 blocks
Point 322 only matches 6 blocks
Point 348 only matches 5 blocks
Point 352 only matches 7 blocks
Point 390 only matches 7 blocks
Point 392 only matches 6 blocks
Point 442 only matches 5 blocks
Point 443 only matches 7 blocks
Point 448 only matches 6 blocks
Point 465 only matches 6 blocks
Point 475 only matches 6 blocks
Point 516 only matches 6 blocks
Point 531 only matches 6 blocks
Point 532 only matches 6 blocks
Point 622 only matches 7 blocks
Point 639 only matches 5 blocks
Point 705 only matches 7 blocks
Point 739 only matches 6 blocks
Point 847 only matches 6 blocks
Point 870 only matches 7 blocks
Point 877 only matches 7 blocks
Average number of blocks:  38.48432798395186 min:  8 max:  89
Number of items:  7976


### Small partner proteins
For small partner proteins (10-30 residues), we keep all the blocks and retrieve atthe residue level.

In [ ]:
items = []
for _, row in ppis[(ppis['num_blocks'] >= 10) & (ppis['num_blocks'] <= 30)].iterrows():
    data = blocks_to_data(row['blocks'])
    data['block_to_pdb_indexes'] = json.dumps({k: v for k, v in zip(range(1, len(row['blocks']) + 1), row['pdb_indexes'])})
    data['id'] = f"{row['PDBProtProt']}_{row['Chain_Partner']}"
    items.append(data)

pd.DataFrame(items).to_parquet('PPI_Partner_small_proteins.parquet')

In [28]:
items = []
for _, row in ppis[(ppis['num_blocks'] < 10)].iterrows():
    data = blocks_to_data(row['blocks'])
    data['block_to_pdb_indexes'] = json.dumps({k: v for k, v in zip(range(1, len(row['blocks']) + 1), row['pdb_indexes'])})
    data['id'] = f"{row['PDBProtProt']}_{row['Chain_Partner']}"
    items.append(data)
print("Number of items: ", len(items))
pd.DataFrame(items).to_parquet('PPI_Partner_very_small_proteins.parquet')

Number of items:  15
